# Visa postcode-area spend to ITL1

This notebook is an alternative execution channel for the `visa-itl1` package. It exposes each pipeline stage so that the input schema, commercial-property selection, matching, weights, allocation and reconciliation can be inspected before outputs are written.

The primary estimate:

1. filters property-level AddressBase records using `classification_code`;
2. matches commercial-property full postcodes to NSPL;
3. calculates commercial-property shares by postcode area and ITL1;
4. constructs postcode-area Visa control totals; and
5. allocates those totals fractionally to ITL1.


## 1. Imports and package setup

Run the notebook from the package repository root. If the package has not been installed, the cell adds `src/` to the Python path. For a permanent editable installation, use `python -m pip install -e .` in the project environment.


In [ ]:
from pathlib import Path
import sys
import json
import yaml
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from visa_itl1.io import read_table, write_table
from visa_itl1.nspl import prepare_nspl
from visa_itl1.addressbase import prepare_commercial_properties
from visa_itl1.weights import build_postcode_area_itl1_weights
from visa_itl1.visa import prepare_visa, create_area_controls
from visa_itl1.allocation import allocate_visa_area_controls
from visa_itl1.diagnostics import visa_geography_coverage, match_summary

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 2. File locations

Change these paths to the actual files. CSV and Parquet inputs are supported by the package.


In [ ]:
NSPL_PATH = PROJECT_ROOT / "data" / "nspl.parquet"
ADDRESSBASE_PATH = PROJECT_ROOT / "data" / "addressbase.parquet"
VISA_PATH = PROJECT_ROOT / "data" / "visa.parquet"
CONFIG_PATH = PROJECT_ROOT / "config" / "example.yml"
OUTPUT_DIR = PROJECT_ROOT / "outputs_notebook"

for label, path in {
    "NSPL": NSPL_PATH,
    "AddressBase": ADDRESSBASE_PATH,
    "Visa": VISA_PATH,
    "Configuration": CONFIG_PATH,
}.items():
    print(f"{label:14}: {path} | exists={path.exists()}")


## 3. Load and inspect configuration

Before continuing, ensure that:

- the AddressBase commercial classification rule is the approved rule rather than the illustrative `C` prefix;
- `region_names` contains the approved `rgn_code` to ITL1-name dictionary;
- Visa spend columns and non-geographic dimensions are correct; and
- a property identifier is configured if AddressBase requires property-level deduplication.


In [ ]:
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print(yaml.safe_dump(cfg, sort_keys=False))

nspl_cfg = cfg["nspl"]
address_cfg = cfg["addressbase"]
visa_cfg = cfg["visa"]
allocation_cfg = cfg["allocation"]
region_names = cfg["region_names"]


## 4. Load source tables


In [ ]:
nspl_raw = read_table(NSPL_PATH)
addressbase_raw = read_table(ADDRESSBASE_PATH)
visa_raw = read_table(VISA_PATH)

print("NSPL shape:", nspl_raw.shape)
print("AddressBase shape:", addressbase_raw.shape)
print("Visa shape:", visa_raw.shape)

display(nspl_raw.head())
display(addressbase_raw.head())
display(visa_raw.head())


## 5. Prepare the NSPL lookup

This stage normalises full postcodes, checks that a postcode does not map to multiple ITL1 codes, validates every observed code against the supplied dictionary, and derives postcode area.


In [ ]:
nspl_lookup, nspl_invalid = prepare_nspl(
    nspl_raw,
    postcode_column=nspl_cfg["postcode_column"],
    region_code_column=nspl_cfg["region_code_column"],
    region_names=region_names,
)

print("Valid NSPL lookup rows:", len(nspl_lookup))
print("Invalid NSPL rows:", len(nspl_invalid))
display(nspl_lookup.head())
if not nspl_invalid.empty:
    display(nspl_invalid.head())


## 6. Select commercial AddressBase properties

The classification summary should be reviewed carefully. If repeated AddressBase rows represent the same property, configure a unique property identifier and enable deduplication before using the counts as weights.


In [ ]:
commercial_properties, classification_summary, addressbase_invalid = (
    prepare_commercial_properties(
        addressbase_raw,
        postcode_column=address_cfg["postcode_column"],
        classification_column=address_cfg["classification_column"],
        classification_rule=address_cfg["classification_filter"],
        property_id_column=address_cfg.get("property_id_column"),
        deduplicate_properties=address_cfg.get("deduplicate_properties", False),
    )
)

print("Selected commercial-property rows:", len(commercial_properties))
print("Commercial rows with invalid postcode:", len(addressbase_invalid))
display(classification_summary.head(30))
display(commercial_properties.head())


## 7. Build postcode-area-to-ITL1 weights

Weights are calculated from matched commercial-property counts. Within each postcode area, the weights must sum to one. Cross-boundary postcode areas receive more than one ITL1 row.


In [ ]:
weights, unmatched_addressbase, weight_diagnostics = (
    build_postcode_area_itl1_weights(
        commercial_properties,
        nspl_lookup,
        tolerance=allocation_cfg.get("weight_tolerance", 1e-10),
    )
)

addressbase_match_summary = match_summary(
    total=len(commercial_properties),
    matched=len(commercial_properties) - len(unmatched_addressbase),
    label="AddressBase to NSPL",
)

print("Weight rows:", len(weights))
print("Postcode areas represented:", weights["postcode_area"].nunique())
print("Cross-boundary postcode areas:", weights.loc[weights["cross_boundary"], "postcode_area"].nunique())
display(addressbase_match_summary)
display(weight_diagnostics.head())
display(weights.loc[weights["cross_boundary"]].head(30))


## 8. Prepare Visa geography and spend values

The package validates the postcode components independently and constructs district, sector and full-postcode diagnostic keys. The primary estimator still uses postcode area as the control geography.

Configured suppression tokens become missing values, not zero.


In [ ]:
prepared_visa, visa_spend_quality, visa_level_summary = (
    prepare_visa(
        visa_raw,
        postcode_level_column=(
            visa_cfg["postcode_level_column"]
        ),
        postcode_code_column=(
            visa_cfg["postcode_code_column"]
        ),
        postcode_levels=(
            visa_cfg["postcode_levels"]
        ),
        spend_columns=(
            visa_cfg["spend_columns"]
        ),
        dimensions=(
            visa_cfg.get("dimensions", [])
        ),
        suppressed_tokens=(
            visa_cfg.get(
                "suppressed_tokens",
                [],
            )
        ),
        unknown_level_policy=(
            visa_cfg.get(
                "unknown_level_policy",
                "error",
            )
        ),
    )
)

display(visa_spend_quality)
display(visa_level_summary)

display(
    prepared_visa[
        [
            visa_cfg["postcode_level_column"],
            visa_cfg["postcode_code_column"],
            "postcode_level_key",
            "postcode_code_clean",
            "geography_type_detected",
            "geography_status",
            "postcode_area_key",
            "postcode_district_key",
            "postcode_sector_key",
            "full_postcode_key",
        ]
    ].head(30)
)

visa_coverage = visa_geography_coverage(prepared_visa)

display(visa_spend_quality)
display(visa_coverage)
display(prepared_visa.head())


In [ ]:
visa_geography_status = (
    prepared_visa.groupby(
        [
            "postcode_level_key",
            "geography_type_detected",
            "geography_status",
        ],
        dropna=False,
    )
    .size()
    .rename("record_count")
    .reset_index()
    .sort_values(
        [
            "postcode_level_key",
            "geography_status",
            "record_count",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
)

display(visa_geography_status)

## 9. Construct postcode-area Visa controls

Controls are grouped by the configured non-geographic dimensions, such as period and MCG. `min_count=1` ensures an entirely missing or suppressed group remains missing instead of becoming zero.


In [ ]:
area_controls = create_area_controls(
    prepared_visa,
    area_level_label=(
        visa_cfg["postcode_levels"]["area"]
    ),
    spend_columns=(
        visa_cfg["spend_columns"]
    ),
    dimensions=(
        visa_cfg.get("dimensions", [])
    ),
)

print(
    "Postcode-area control rows:",
    len(area_controls),
)

display(area_controls.head(30))


In [ ]:
parent_geography_rows = prepared_visa.loc[
    prepared_visa[
        "geography_status"
    ].eq("parent_at_child_level")
].copy()

print(
    "Parent-geography codes reported at a child level:",
    len(parent_geography_rows),
)

display(
    parent_geography_rows[
        [
            *visa_cfg.get("dimensions", []),
            visa_cfg["postcode_level_column"],
            visa_cfg["postcode_code_column"],
            "geography_type_detected",
            "postcode_area_key",
            "postcode_district_key",
            *visa_cfg["spend_columns"],
        ]
    ].head(50)
)

## 10. Allocate controls to ITL1 and reconcile

Only postcode areas with a valid AddressBase weight contribute to the matched ITL1 estimate. Unmatched controls are retained separately and quantified in the reconciliation table.


In [ ]:
visa_itl1, unmatched_visa, reconciliation = allocate_visa_area_controls(
    area_controls,
    weights,
    spend_columns=visa_cfg["spend_columns"],
    dimensions=visa_cfg.get("dimensions", []),
    unmatched_policy=allocation_cfg.get("unmatched_policy", "retain"),
    reconciliation_tolerance=allocation_cfg.get(
        "reconciliation_tolerance", 1e-8
    ),
)

print("ITL1 output rows:", len(visa_itl1))
print("Unmatched Visa control rows:", len(unmatched_visa))
display(reconciliation)
display(visa_itl1.head(30))
if not unmatched_visa.empty:
    display(unmatched_visa.head(30))


In [ ]:
all_level_key = (
    str(visa_cfg["postcode_levels"]["all"])
    .strip()
    .upper()
)

all_level_rows = prepared_visa.loc[
    prepared_visa[
        "postcode_level_key"
    ].eq(all_level_key)
].copy()

group_dimensions = visa_cfg.get(
    "dimensions",
    [],
)

all_level_totals = (
    all_level_rows.groupby(
        group_dimensions,
        dropna=False,
        as_index=False,
    )[visa_cfg["spend_columns"]]
    .sum(min_count=1)
    if group_dimensions
    else all_level_rows[
        visa_cfg["spend_columns"]
    ].sum(min_count=1).to_frame().T
)

display(all_level_totals)

## 11. Optional grouped reconciliation

This check compares matched postcode-area controls with ITL1 allocations within every configured dimension combination. It supplements the package's overall reconciliation.


In [ ]:
dimensions = visa_cfg.get("dimensions", [])
spend_columns = visa_cfg["spend_columns"]
matched_areas = set(weights["postcode_area"])
matched_controls = area_controls[
    area_controls["postcode_area_key"].isin(matched_areas)
].copy()

if dimensions:
    input_by_group = matched_controls.groupby(
        dimensions, dropna=False, as_index=False
    )[spend_columns].sum(min_count=1)

    allocated_columns = [f"{c}_allocated" for c in spend_columns]
    output_by_group = visa_itl1.groupby(
        dimensions, dropna=False, as_index=False
    )[allocated_columns].sum(min_count=1)

    grouped_reconciliation = input_by_group.merge(
        output_by_group, on=dimensions, how="outer", validate="one_to_one"
    )
    for spend in spend_columns:
        grouped_reconciliation[f"{spend}_difference"] = (
            grouped_reconciliation[f"{spend}_allocated"]
            - grouped_reconciliation[spend]
        )
else:
    grouped_reconciliation = reconciliation.copy()

display(grouped_reconciliation.head(30))


## 12. Write notebook outputs

This mirrors the package pipeline while also writing the grouped reconciliation created above.


In [ ]:
main_dir = OUTPUT_DIR / "main"
lookup_dir = OUTPUT_DIR / "lookups"
diagnostic_dir = OUTPUT_DIR / "diagnostics"
reconciliation_dir = OUTPUT_DIR / "reconciliation"
for directory in [main_dir, lookup_dir, diagnostic_dir, reconciliation_dir]:
    directory.mkdir(parents=True, exist_ok=True)

write_table(visa_itl1, main_dir / "visa_itl1.parquet")
write_table(nspl_lookup, lookup_dir / "nspl_postcode_itl1.parquet")
write_table(weights, lookup_dir / "postcode_area_itl1_weights.parquet")
write_table(classification_summary, diagnostic_dir / "addressbase_classification_summary.csv")
write_table(addressbase_match_summary, diagnostic_dir / "addressbase_nspl_match_summary.csv")
write_table(nspl_invalid, diagnostic_dir / "nspl_invalid_rows.parquet")
write_table(addressbase_invalid, diagnostic_dir / "addressbase_invalid_postcodes.parquet")
write_table(unmatched_addressbase, diagnostic_dir / "addressbase_unmatched_nspl.parquet")
write_table(weight_diagnostics, diagnostic_dir / "weight_validation.csv")
write_table(weights.loc[weights["cross_boundary"]], diagnostic_dir / "cross_boundary_postcode_areas.csv")
write_table(visa_coverage, diagnostic_dir / "visa_geography_coverage.csv")
write_table(visa_spend_quality, diagnostic_dir / "visa_spend_quality.csv")
write_table(unmatched_visa, diagnostic_dir / "visa_unmatched_postcode_areas.parquet")
write_table(reconciliation, reconciliation_dir / "reconciliation_overall.csv")
write_table(grouped_reconciliation, reconciliation_dir / "reconciliation_by_dimensions.csv")
write_table(
    visa_level_summary,
    diagnostic_dir
    / "visa_postcode_level_summary.csv",
)

write_table(
    visa_geography_status,
    diagnostic_dir
    / "visa_geography_status.csv",
)

write_table(
    parent_geography_rows,
    diagnostic_dir
    / "visa_parent_geography_at_child_level.parquet",
)

write_table(
    all_level_totals,
    reconciliation_dir
    / "visa_all_level_totals.csv",
)

(OUTPUT_DIR / "run_config.yml").write_text(
    yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8"
)
manifest = {
    "execution_channel": "notebook",
    "output_files": sorted(
        str(path.relative_to(OUTPUT_DIR))
        for path in OUTPUT_DIR.rglob("*")
        if path.is_file()
    ),
}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print(f"Outputs written to: {OUTPUT_DIR.resolve()}")
for path in manifest["output_files"]:
    print(" -", path)


## 13. Optional one-call package execution

The staged cells above are preferable during development and review. Once the configuration is stable, the package's complete pipeline can also be called directly from the notebook.


In [ ]:
# from visa_itl1.pipeline import run_pipeline
#
# run_pipeline(
#     nspl_path=str(NSPL_PATH),
#     addressbase_path=str(ADDRESSBASE_PATH),
#     visa_path=str(VISA_PATH),
#     config_path=str(CONFIG_PATH),
#     output_dir=str(PROJECT_ROOT / "outputs_pipeline_call"),
# )
